In [1]:
import json
import os
import re
import pandas as pd

In [16]:
# with open('pydantic_llama3_8b_judge_processed.jsonl', 'r') as f:
#     llama3_8b = [json.loads(line) for line in f]

# with open('pydantic_llama3_8b_judge_processed_baseline.jsonl', 'r') as f:
#     llama3_8b_baseline = [json.loads(line) for line in f]
    
wk_expert = pd.read_csv('wenkai_subsample_expert_annotation_4_n_each_feature.csv')
ly_expert = pd.read_csv('lynnette_subsample_expert_annotation_4_n_each_feature.csv')
# llama3_8b_baseline_df = pd.DataFrame(llama3_8b_baseline)
# llama3_8b_df = pd.DataFrame(llama3_8b)

In [17]:
wk_expert_json = wk_expert.to_dict(orient='records')
ly_expert_json = ly_expert.to_dict(orient='records')
# wk_expert_json_str = json.dumps(wk_expert_json, ensure_ascii=False, indent=2)

In [27]:
# 找到wk/ly_expert_json共同为1的samples，并得到所有8个features都有过1的最小subsamples（尽量让每个sample覆盖更多features）
# 最终final_samples只保存原始信息（wk/ly_expert_json中任意一个即可，因为内容一样）

features = [
    "1gamemove.yes", "2reasoning.yes", "3rapport.yes", "3a_apologies.yes",
    "3a_compliment.yes", "3a_personalthoughts.yes",
    "3a_reassurance.yes", "4shareinformation.yes",
]

wk_df = pd.DataFrame(wk_expert_json)
ly_df = pd.DataFrame(ly_expert_json)

# 合并，按Input.full_text对齐
merged = pd.merge(
    wk_df[["Input.full_text"] + features],
    ly_df[["Input.full_text"] + features],
    on="Input.full_text",
    suffixes=("_wk", "_ly")
)

# 只保留两人都为1的feature
def get_joint_1_feats(row):
    feats = []
    for feat in features:
        if row[f"{feat}_wk"] == 1 and row[f"{feat}_ly"] == 1:
            feats.append(feat)
    return feats

merged["joint_1_feats"] = merged.apply(get_joint_1_feats, axis=1)
# 只保留至少有一个feature为1的样本
joint_samples = merged[merged["joint_1_feats"].apply(len) > 0].copy()

# 贪心法：每次选能覆盖最多未覆盖feature的样本
covered_feats = set()
final_samples = []
while len(covered_feats) < len(features):
    # 计算每个样本能覆盖多少未覆盖的feature
    joint_samples["uncovered"] = joint_samples["joint_1_feats"].apply(lambda feats: set(feats) - covered_feats)
    # 只考虑能覆盖新feature的样本
    candidates = joint_samples[joint_samples["uncovered"].apply(len) > 0]
    if len(candidates) == 0:
        break
    # 选能覆盖最多新feature的样本
    idx = candidates["uncovered"].apply(len).idxmax()
    row = candidates.loc[idx]
    # 保存Input.full_text
    input_text = row["Input.full_text"]
    # 从原始json中找到完整信息（wk/ly任选一个即可）
    orig_info = next(item for item in wk_expert_json if item["Input.full_text"] == input_text)
    final_samples.append(orig_info)
    covered_feats.update(row["uncovered"])
    # 移除已选样本
    joint_samples = joint_samples.drop(idx)

# print("最小subsample覆盖8个features的样本（只包含原始信息）:")
# for sample in final_samples:
#     print("-", sample)

In [28]:
with open('few_shot_samples.json', 'w') as f:
    json.dump(final_samples, f, ensure_ascii=False, indent=2)

In [35]:
# Check for duplicate Input.full_text
duplicate_texts = ly_expert[ly_expert['Input.full_text'].duplicated()]
if len(duplicate_texts) > 0:
    print(f"Found {len(duplicate_texts)} duplicate Input.full_text entries:")
    print(duplicate_texts[['Input.full_text']])
else:
    print("No duplicate Input.full_text found")
ly_expert

Found 2 duplicate Input.full_text entries:
                                       Input.full_text
93   Yeah I didnt believe that turkey wanted to go ...
118  I imagine France would be open to a deal that ...


,Input.full_text,1gamemove.yes,2reasoning.yes,3rapport.yes,3a_apologies.yes,3a_compliment.yes,3a_personalthoughts.yes,3a_reassurance.yes,4shareinformation.yes
0,Happy to aid in the fight against germany and ...,0,0,1,0,0,0,0,0
1,I'm going to be hit or miss with connectivity ...,0,0,0,0,0,1,0,0
2,"And if so, how do you feel about it?",0,0,1,0,0,0,0,0
3,I'd like to hold it for at least one more seas...,1,0,0,0,0,0,0,0
4,I have so much to offer besides just bedside a...,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
123,If you agree to a ceasefire I promise to pull ...,1,0,0,0,0,0,0,0
124,I have another proposal if youre interested,0,0,0,0,0,1,0,0
125,What is the out of game thing?,0,0,0,0,0,1,0,0
126,"Also, France has offered a DMZ in the English ...",1,0,0,0,0,0,0,1


In [36]:
import numpy as np
import pandas as pd
from statsmodels.stats.inter_rater import fleiss_kappa
from typing import List, Dict, Tuple, Optional

DEFAULT_CATEGORIES = [
    "1gamemove.yes", "2reasoning.yes", "3rapport.yes", "3a_apologies.yes",
    "3a_compliment.yes", "3a_personalthoughts.yes",
    "3a_reassurance.yes", "4shareinformation.yes",
]

# ---------- helper: yes/no → 0/1 -------------------------------------
def _to_int(label) -> int:
    if pd.isna(label) or label is None:
        return 0
    if isinstance(label, bool):
        return int(label)
    return 1 if str(label).strip().lower() in {"yes", "true", "1"} else 0

# ---------- helper: DataFrame → {full_text: row_dict} ----------------
def _index_by_text(df: pd.DataFrame, *, keep: str = "last"):
    """
    将 DataFrame 索引成 {full_text: row_dict}。
    若 full_text 重复则按 keep 参数保留，并返回:
        dict_rows, removed_count
    """
    before = len(df)
    dedup_df = df.drop_duplicates(subset="Input.full_text", keep=keep)
    removed = before - len(dedup_df)

    if removed:
        sample = dedup_df["Input.full_text"].head(1).iloc[0]
        print(f"[WARN] {removed} duplicates dropped (keep='{keep}'). "
              f"Example kept text: {sample!r}")

    return (
        dedup_df.set_index("Input.full_text").to_dict(orient="index"),
        removed,
    )

def build_rating_matrix(df_model, df_expert1, df_expert2, categories=None):
    cats = categories or DEFAULT_CATEGORIES

    # 1) 三方共有文本
    common_texts = sorted(
        set(df_model["Input.full_text"])
        & set(df_expert1["Input.full_text"])
        & set(df_expert2["Input.full_text"])
    )

    # 2) 建立索引 + 获取重复计数
    (d_model,  rm_model),  \
    (d_exp1,   rm_exp1),   \
    (d_exp2,   rm_exp2) = (_index_by_text(df) for df in (df_model,
                                                         df_expert1,
                                                         df_expert2))

    # 3) 构造投票矩阵
    rows = []
    for txt in common_texts:
        for cat in cats:
            votes = [0, 0]
            for d in (d_model, d_exp1, d_exp2):
                votes[_to_int(d[txt][cat])] += 1
            rows.append(votes)

    ratings = np.asarray(rows, dtype=int)
    removed_summary = {"model": rm_model, "expert1": rm_exp1, "expert2": rm_exp2}
    return ratings, removed_summary


def compute_fleiss_kappa(df_model, df_expert1, df_expert2,
                         categories=None, verbose=True):
    cats = categories or DEFAULT_CATEGORIES

    ratings, removed = build_rating_matrix(df_model, df_expert1, df_expert2, cats)
    overall = fleiss_kappa(ratings)

    kappa_by_cat = {}
    block = len(cats)
    for i, cat in enumerate(cats):
        kappa_by_cat[cat] = fleiss_kappa(ratings[i::block])

    if verbose:
        print(f"Overall Fleiss' κ: {overall:.3f}")
        for cat, k in kappa_by_cat.items():
            print(f"{cat:25s}: κ = {k:.3f}")

        total_removed = sum(removed.values())
        print(f"\n▶️  Duplicate texts removed → {total_removed} "
              f"(model={removed['model']}, expert1={removed['expert1']}, "
              f"expert2={removed['expert2']})")

    # 返回第三项：重复计数
    return overall, kappa_by_cat, removed


In [37]:
overall_k, per_cat_k, dup_counts = compute_fleiss_kappa(
    llama3_8b_df, wk_expert, ly_expert
)

# 如果只想拿到删除数量
# print("总共删除了", sum(dup_counts.values()), "条重复文本")


[WARN] 2 duplicates dropped (keep='last'). Example kept text: 'Happy to aid in the fight against germany and france as well'
[WARN] 2 duplicates dropped (keep='last'). Example kept text: 'Happy to aid in the fight against germany and france as well'
[WARN] 2 duplicates dropped (keep='last'). Example kept text: 'Happy to aid in the fight against germany and france as well'
Overall Fleiss' κ: 0.303
1gamemove.yes            : κ = 0.374
2reasoning.yes           : κ = 0.306
3rapport.yes             : κ = 0.223
3a_apologies.yes         : κ = 0.478
3a_compliment.yes        : κ = -0.027
3a_personalthoughts.yes  : κ = 0.178
3a_reassurance.yes       : κ = -0.171
4shareinformation.yes    : κ = 0.310

▶️  Duplicate texts removed → 6 (model=2, expert1=2, expert2=2)


In [38]:
overall_k, per_cat_k, dup_counts = compute_fleiss_kappa(
    llama3_8b_baseline_df, wk_expert, ly_expert
)

# 如果只想拿到删除数量
# print("总共删除了", sum(dup_counts.values()), "条重复文本")


[WARN] 2 duplicates dropped (keep='last'). Example kept text: 'Happy to aid in the fight against germany and france as well'
[WARN] 2 duplicates dropped (keep='last'). Example kept text: 'Happy to aid in the fight against germany and france as well'
[WARN] 2 duplicates dropped (keep='last'). Example kept text: 'Happy to aid in the fight against germany and france as well'
Overall Fleiss' κ: 0.201
1gamemove.yes            : κ = 0.353
2reasoning.yes           : κ = 0.208
3rapport.yes             : κ = 0.080
3a_apologies.yes         : κ = -0.104
3a_compliment.yes        : κ = -0.086
3a_personalthoughts.yes  : κ = 0.241
3a_reassurance.yes       : κ = -0.002
4shareinformation.yes    : κ = 0.249

▶️  Duplicate texts removed → 6 (model=2, expert1=2, expert2=2)
